### ⚠️ 실행 안내 및 데이터 공지

1. 본 프로젝트는 Kaggle Starbucks Rewards 데이터를 기반으로 작성되었습니다.  
   ( transcript.csv, profile.csv는 개인정보 포함으로 제외)

2. 전체 실행을 위해 아래 데이터 다운로드 후 `data/raw/` 경로에 추가해야 합니다.  
   [[Kaggle Starbucks Rewards 데이터]](https://www.kaggle.com/datasets/ihormuliar/starbucks-customer-data)

3. 본 노트북 실행 시  
   `data/processed/prep_master_table.csv`가 생성됩니다.

4. 이후 노트북(02~05)은 해당 파일을 기반으로 실행됩니다.

# 01. 데이터 준비 — 마스터 테이블 구축

**분석 목적:** 3개 원본 데이터를 통합하여 Receive → View → Complete 퍼널 추적이 가능한 분석 환경 구성  
**데이터:** 고객 17,000명 · 이벤트 306,534건  
**산출물:** `prep_master_table.csv`

**처리 순서:**

| 단계 | 처리 | 설명 |
|------|------|------|
| 1 | profile 전처리 | 118세(미기재) 제거 · 가입기간·연령·소득 범주화 |
| 2 | transcript 전처리 | value 컬럼(딕셔너리) 파싱 → offer_id, amount 분리 |
| 3 | portfolio 전처리 | channels(리스트) → 이메일·모바일·소셜·웹 더미변수화 |
| 4 | 마스터 테이블 구축 | 3-way merge: transcript + profile + portfolio |
| 5 | 이상치 제거 | IQR 기준 amount 정제 (0.05 ~ 41달러) |

---

## 0. 환경 설정

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

## 1. 데이터 로드

In [2]:
# 데이터 로드

portfolio = pd.read_csv("../data/raw/portfolio.csv")
profile = pd.read_csv("../data/raw/profile.csv")
transcript = pd.read_csv("../data/raw/transcript.csv")

## 2. profile 전처리

**문제:** 나이 118세는 스타벅스가 임의 지정한 미기재 값 (2,175건)  
→ 성별·소득 결측치와 100% 일치 → 분석 대상에서 제외

---

### 2-1. 결측치 확인

In [3]:
# profile 복사 및 가입일 datetime 변환
profile11 = profile.copy()
profile11["became_member_on"] = pd.to_datetime(profile11["became_member_on"], format="%Y%m%d").dt.normalize()

### 2-2. 가입 기간 범주화

In [4]:
# 가입 경과 연수 계산 및 범주화

# 가장 최근 가입한 날짜 기준
reference_date = profile11['became_member_on'].max()

# 가입 경과 연수 (년 단위)
profile11['membership_years'] = (reference_date - profile11['became_member_on']).dt.days / 365.25

# 범주화
profile11['membership_g'] = np.select(
    [
        profile11['membership_years'] < 1,
        (profile11['membership_years'] >= 1) & (profile11['membership_years'] < 3),
        profile11['membership_years'] >= 3
    ],
    [
        'Under 1yr',
        '1~3yr',
        '3yr+'
    ],
    default='Unknown'
)

profile11['membership_g'].value_counts()


[회원연수 범주 분포 확인]


membership_g
Under 1yr    8706
1~3yr        6912
3yr+         1382
Name: count, dtype: int64

### 2-3. 성별·연령·소득 범주화

| 변수 | 범주 | 기준 |
|------|------|------|
| gender_g | F / M / O / Unknown | 원본값 유지, NaN → Unknown |
| age_g | 10세 단위 6구간 | 118세 → Unknown |
| income_g | Low-End ~ VIP | 5만 달러 단위 구간 |
| membership_g | Under 1yr / 1~2yr / 3yr+ | 가입 경과 연수 기준 |

In [5]:
# 성별 범주화

profile12 = profile11.copy()

profile12["gender_g"] = (
    profile12["gender"]
    .replace({"F": "F", "M": "M", "O": "O"})
    .fillna("Unknown")
)


profile12["gender_g"].value_counts()

[성별 범주 분포 확인]


gender_g
M          8484
F          6129
Unknown    2175
O           212
Name: count, dtype: int64

In [6]:
# 연령 범주화 — 118세는 결측 처리

profile12["age_g"] = np.select(
    [
        profile12["age"] == 118,
        profile12["age"] <= 24,
        profile12["age"].between(25, 34),
        profile12["age"].between(35, 44),
        profile12["age"].between(45, 54),
        profile12["age"].between(55, 64),
        profile12["age"] >= 65
    ],
    [
        "Unknown",
        "under_24",
        "25-34",
        "35-44",
        "45-54",
        "55-64",
        "65+"
    ],
    default="Unknown"
)


profile12["age_g"].value_counts()

[연령 범주 분포 확인]


age_g
65+         4266
55-64       3421
45-54       3013
Unknown     2175
35-44       1869
25-34       1380
under_24     876
Name: count, dtype: int64

In [7]:
# 소득 5구간 범주화

income_bins = [0, 50000, 65000, 80000, 100000, float("inf")]
income_labels = [
    "Low-End",
    "Mid-Low",
    "Mid-High",
    "High-End",
    "VIP"
]

profile12["income_g"] = pd.cut(
    profile12["income"],
    bins=income_bins,
    labels=income_labels,
    right=False  # 구간 왼쪽 포함
)

# 결측치 → Unknown
profile12["income_g"] = (
    profile12["income_g"]
    .cat.add_categories("Unknown")
    .fillna("Unknown")
)


profile12["income_g"].value_counts()

[수입 범주 분포 확인]


income_g
Mid-Low     3863
Low-End     3781
Mid-High    3464
High-End    2624
Unknown     2175
VIP         1093
Name: count, dtype: int64

## 3. transcript 전처리

**문제:** value 컬럼이 문자열 형태의 딕셔너리 → 직접 접근 불가  
→ `ast.literal_eval` 파싱 후 컬럼으로 분리  
→ `offer id`(띄어쓰기)와 `offer_id`(언더바) 두 형태 통합

---

In [8]:
# value 컬럼 파싱 — ast.literal_eval로 딕셔너리 변환
import ast

transcript12 = transcript.copy()

transcript12['value'] = transcript12['value'].apply(ast.literal_eval)

## 펼치기
value_df = pd.json_normalize(transcript12['value'])
value_df

,offer id,amount,offer_id,reward
0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,NaN
1,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN
2,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,NaN
3,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN
4,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN
...,...,...,...,...
306529,NaN,1.59,NaN,NaN
306530,NaN,9.53,NaN,NaN
306531,NaN,3.61,NaN,NaN
306532,NaN,3.53,NaN,NaN


In [9]:
# offer id 컬럼명 통일 — 원본 띄어쓰기 차이 처리
value_df['offer_id'] = value_df['offer_id'].fillna(value_df['offer id'])
value_df = value_df.drop(columns=['offer id'])
value_df

,amount,offer_id,reward
0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN
1,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN
2,NaN,2906b810c7d4411798c6938adc9daaa5,NaN
3,NaN,fafdcd668e3743c1bb461111dcafc2a4,NaN
4,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN
...,...,...,...
306529,1.59,NaN,NaN
306530,9.53,NaN,NaN
306531,3.61,NaN,NaN
306532,3.53,NaN,NaN


In [10]:
# 파싱 결과를 transcript에 병합
transcript12 = pd.concat([transcript12.drop(columns=['Unnamed: 0','value']), value_df], axis=1)
transcript12

,person,event,time,amount,offer_id,reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN
...,...,...,...,...,...,...
306529,b3a1272bc9904337b331bf348c3e8c17,transaction,714,1.59,NaN,NaN
306530,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,714,9.53,NaN,NaN
306531,a00058cf10334a308c68e7631c529907,transaction,714,3.61,NaN,NaN
306532,76ddbd6576844afe811f1a3c0fbb5bec,transaction,714,3.53,NaN,NaN


## 4. portfolio 전처리

channels 컬럼 리스트(`[email, mobile, web]`) → 채널별 더미변수  
→ 오퍼별 발송 채널 조합 분석에 사용

---

In [11]:
# portfolio 복사
portfolio13 = portfolio.copy()

In [12]:
portfolio13['channels'] = portfolio13['channels'].apply(ast.literal_eval)
chnl_type_change = type(portfolio13.loc[0, 'channels'])


channels 타입 변경 후 :  <class 'list'>


In [13]:
# channels 리스트 → 더미변수 변환
channels_dummies = portfolio13['channels'].str.join('|').str.get_dummies()
channels_dummies

,email,mobile,social,web
0,1,1,1,0
1,1,1,1,1
2,1,1,0,1
3,1,1,0,1
4,1,0,0,1
5,1,1,1,1
6,1,1,1,1
7,1,1,1,0
8,1,1,1,1
9,1,1,0,1


In [14]:
# 더미변수를 portfolio에 병합
portfolio13 = pd.concat([portfolio13, channels_dummies], axis=1)

## 5. 마스터 테이블 구축 (3-way merge)

**병합 순서:** transcript → profile (고객 ID 기준) → portfolio (오퍼 ID 기준)  
**Unknown 제거:** 성별·소득 미기재 고객 제거 → 인구통계 분석 기반 확보

---

In [15]:
# transcript ← profile 병합 (고객 ID 기준)
df14 = transcript12.merge(
    profile12, 
    left_on='person', 
    right_on='id', 
    how='left'
)

## 중복 컬럼 제거
df14 = df14.drop(columns=['id', 'Unnamed: 0'])

## 결측치 확인
display(df14.isna().sum())

[transcript + profile 데이터셋 결측치 확인]


person                   0
event                    0
time                     0
amount              167581
offer_id            138953
reward              272955
gender               33772
age                      0
became_member_on         0
income               33772
membership_years         0
membership_g             0
gender_g                 0
age_g                    0
income_g                 0
dtype: int64

In [16]:
# 성별·소득 미기재(Unknown) 제거
df14_clean = df14.drop(
    df14[
        (df14["gender_g"] == "Unknown") |
        (df14["income_g"] == "Unknown")
    ].index
).copy()

display(df14_clean.isna().sum())

[Unknown 제거 후 결측치 확인]


person                   0
event                    0
time                     0
amount              148805
offer_id            123957
reward              240318
gender                   0
age                      0
became_member_on         0
income                   0
membership_years         0
membership_g             0
gender_g                 0
age_g                    0
income_g                 0
dtype: int64

In [17]:
# ← portfolio 병합 (오퍼 ID 기준)
mt = df14_clean.merge(
    portfolio13, 
    left_on='offer_id',
    right_on='id',
    how='left',
    suffixes=('', '_offer')
)

## 중복 컬럼 제거
mt = mt.drop(columns=['id', 'Unnamed: 0'])
mt

,person,event,time,amount,offer_id,reward,gender,age,became_member_on,income,...,income_g,reward_offer,channels,difficulty,duration,offer_type,email,mobile,social,web
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,F,75,2017-05-09,100000.0,...,VIP,5.0,"[web, email, mobile]",5.0,7.0,bogo,1.0,1.0,0.0,1.0
1,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,NaN,M,68,2018-04-26,70000.0,...,Mid-High,2.0,"[web, email, mobile]",10.0,7.0,discount,1.0,1.0,0.0,1.0
2,389bc3fa690240e798340f5a15918d5c,offer received,0,NaN,f19421c1d4aa40978ebb69ca19b0e20d,NaN,M,65,2018-02-09,53000.0,...,Mid-Low,5.0,"[web, email, mobile, social]",5.0,5.0,bogo,1.0,1.0,1.0,1.0
3,2eeac8d8feae4a8cad5a6af0499a211d,offer received,0,NaN,3f207df678b143eea3cee63160fa8bed,NaN,M,58,2017-11-11,51000.0,...,Mid-Low,0.0,"[web, email, mobile]",0.0,4.0,informational,1.0,1.0,0.0,1.0
4,aa4862eba776480b8bb9c68455b8c2e1,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,F,61,2017-09-11,57000.0,...,Mid-Low,5.0,"[web, email]",20.0,10.0,discount,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272757,24f56b5e1849462093931b164eb803b5,offer completed,714,NaN,fafdcd668e3743c1bb461111dcafc2a4,2.0,F,48,2017-12-28,80000.0,...,High-End,2.0,"[web, email, mobile, social]",10.0,10.0,discount,1.0,1.0,1.0,1.0
272758,b3a1272bc9904337b331bf348c3e8c17,transaction,714,1.59,NaN,NaN,M,66,2018-01-01,47000.0,...,Low-End,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
272759,68213b08d99a4ae1b0dcb72aebd9aa35,transaction,714,9.53,NaN,NaN,M,52,2018-04-08,62000.0,...,Mid-Low,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
272760,a00058cf10334a308c68e7631c529907,transaction,714,3.61,NaN,NaN,F,63,2013-09-22,52000.0,...,Mid-Low,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. 이상치 제거 — amount

**기준:** IQR(사분위수 범위)  
**정상 범위:** 0.05 ~ 41달러  
→ amount=NaN은 오퍼 이벤트(결제 없음)이므로 유지

---

In [18]:
# duration: day → hour 단위 변환
df21 = mt.copy()
df21["duration_hr"] = df21["duration"] * 24

In [19]:
# amount 이상치 제거 — NaN은 오퍼 이벤트(결제 없음)이므로 유지
df21 = df21[
    df21["amount"].isna() |
    ((df21["amount"] >= 0.05) & (df21["amount"] <= 41))
]

## 7. 저장

`prep_master_table.csv` → 02 ~ 05 노트북 전체에서 공통으로 로드

In [20]:
# 전처리 완료 마스터 테이블 저장
df21.to_csv("../data/processed/prep_master_table.csv", index=False, encoding="utf-8-sig")